In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS
import fiona

# 17 - Merge RegBL and Forest

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `14_OFEV_ecorforest_regions`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.13.csv', index_col=0)

df_regbl.head()

In [ ]:
gdf_forest = gpd.read_file(f"{ROOT}data/raw/14_OFEV_ecorforest_regions/Waldstandortsregionen.shp")

print("CRS:", gdf_forest.crs)

In [ ]:
gdf_forest.head()

## Rename columns

In [ ]:
gdf_forest.columns.tolist()

In [ ]:
COLS_TO_KEEP = [
 'Region_de',
 'Region_fr',
 'Region_it',
 'Region_en',
 'Code',
 'Subcode',
 'Code_Bu',
 'Code_Fi',
 'Shape_Leng',
 'Shape_Area',
 'geometry'
]

gdf_forest = gdf_forest[COLS_TO_KEEP]

# Add 'Forest - ' to the beginning of each column name except 'geometry'
gdf_forest.columns = ['Forest - ' + col if col != 'geometry' else col for col in gdf_forest.columns]

## Convert main DataFrame to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Perform spatial joint

In [ ]:
if "index_right" in df_regbl.columns:
    df_regbl = df_regbl.drop(columns="index_right")

In [ ]:
df_regbl = gpd.sjoin(df_regbl, gdf_forest, how="left", predicate="within")

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.14.csv")